# Dataset inspection

Inspect the pinned Kurt & Çayır CSV release: file integrity, record structure, label support and empty token slots. Reader and validation logic live in `src/turkish_punctuation/data.py`. Original files are preserved.

Source and license findings: `docs/data-audit.md`. This notebook performs data checks and does not run a model.


In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from turkish_punctuation.data import LABELS, find_empty_tokens, read_split, summarize_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

manifest = json.loads((PROJECT_ROOT / "docs/data-manifest.json").read_text())
RAW_DIR = PROJECT_ROOT / manifest["local_directory"]
print("Source revision:", manifest["revision"])

## Read and validate splits

Verify checksums before reading. `read_split` rejects invalid schemas, malformed lists, unequal token/label lengths and unknown labels. It retains source row order and empty strings.


In [ ]:
splits = {}
for entry in manifest["files"]:
    path = RAW_DIR / entry["filename"]
    with path.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    if digest != entry["sha256"] or path.stat().st_size != entry["bytes"]:
        raise ValueError(f"File differs from the pinned release: {path.name}")
    splits[entry["split"]] = read_split(path)

summaries = {name: summarize_split(records) for name, records in splits.items()}
overview = pd.DataFrame.from_dict(summaries, orient="index")
overview.index.name = "split"
overview.drop(columns="label_counts")

## Label support

`token_entries` includes empty slots. Labels describe punctuation after each word; `O` means no mark. Support counts establish the class imbalance before model evaluation.


In [ ]:
label_counts = pd.DataFrame.from_dict(
    {name: summary["label_counts"] for name, summary in summaries.items()},
    orient="index",
).reindex(columns=LABELS)
label_counts.index.name = "split"
display(label_counts)
label_percent = label_counts.div(label_counts.sum(axis=1), axis=0).mul(100)
label_percent.round(3)

## Record inspection

Inspect training records only. `row_id` is the zero-based data-row position within a split, excluding the CSV header. The released word slots precede model subword tokenization.


In [ ]:
SAMPLE_ROW_ID = 0
sample = splits["train"].set_index("row_id").loc[SAMPLE_ROW_ID]
pd.DataFrame(
    {
        "token_index": range(len(sample["tokens"])),
        "token": sample["tokens"],
        "tag": sample["tags"],
    }
)

## Empty token slots

Report each empty slot with its source row and attached punctuation. Inspect the first affected training record in context. The reader retains empty slots; the separate clean-v1 preparation quarantines entire affected records.


In [ ]:
empty_tokens = {name: find_empty_tokens(records) for name, records in splits.items()}
empty_label_counts = (
    pd.DataFrame.from_dict(
        {name: findings["tag"].value_counts().to_dict() for name, findings in empty_tokens.items()},
        orient="index",
    )
    .reindex(columns=LABELS, fill_value=0)
    .fillna(0)
    .astype(int)
)
display(empty_label_counts)
empty_tokens["train"].head(10)

In [ ]:
if not empty_tokens["train"].empty:
    affected_row_id = int(empty_tokens["train"].iloc[0]["row_id"])
    affected = splits["train"].set_index("row_id").loc[affected_row_id]
    print("Training row_id:", affected_row_id)
    display(
        pd.DataFrame(
            {
                "token_index": range(len(affected["tokens"])),
                "token_repr": [repr(token) for token in affected["tokens"]],
                "tag": affected["tags"],
                "is_empty": [token == "" for token in affected["tokens"]],
            }
        )
    )

## Released split overlap

Load the full audit's exact input-overlap statistics. Short repeated expressions alone do not establish document leakage. The original CSVs have no talk IDs, so document-level split independence remains unresolved.


In [ ]:
audit = json.loads((PROJECT_ROOT / "docs/data-audit.json").read_text())
if audit["source_revision"] != manifest["revision"]:
    raise ValueError("Audit and data manifest refer to different releases.")
pd.DataFrame.from_dict(audit["cross_split_overlap"], orient="index")

## Status

All records in this release meet the reader's structural checks. Across the three splits, 651 empty slots carry punctuation labels. This notebook applies no filtering, label reassignment or text normalization.

The separate clean-v1 pipeline now prepares eligible splits with first-subword label alignment and dynamic batch padding. See `configs/preprocessing.json`, `scripts/prepare_data.py` and `docs/data-audit.md`. Exact corpus provenance and dataset permissions remain open items.
